# Transfer Learning Comparison

The main notebook trains a CNN **from scratch**, following the paper. The question a
reader will ask is whether a network **pretrained on ImageNet** does better, and what
that costs in size and speed. This notebook answers it.

| Model | Why it is here |
|---|---|
| **Custom CNN (paper)** | The baseline - 6 conv layers trained from scratch |
| **ResNet50** | The common accuracy reference in the literature |
| **EfficientNetB0** | Designed for better accuracy per parameter |
| **MobileNetV2** | Built for phones - the realistic option for use in a field |

**Protocol.** Each pretrained backbone keeps its ImageNet weights **frozen** and gets a
new 3-class head (global average pooling + softmax). Only the head is trained, on the
**same 70:30 split** as the main notebook, for 10 epochs. That is deliberate: the
comparison is about how useful the pretrained features are, not about who gets the
bigger training budget.

Each backbone also uses **its own preprocessing** - the pixel scaling it was originally
trained with. Feeding a network the wrong range quietly costs accuracy, so this is not
a detail that can be skipped.

**Run this after the main notebook**, which produces `wheat_disease_cnn.keras`. Without
that file the three pretrained models are still compared, just without the baseline row.


## Setup

In [ ]:
import os
import random
import shutil
import time
import hashlib
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## Configuration

Identical to the main notebook, so the split below is byte-for-byte the same partition.

In [ ]:
IMG_SIZE    = (256, 256)
INPUT_SHAPE = IMG_SIZE + (3,)
CLASS_NAMES = ['Brown_Rust', 'Healthy', 'Yellow_Rust']
NUM_CLASSES = len(CLASS_NAMES)

BATCH_SIZE    = 32
LEARNING_RATE = 1e-4
TL_EPOCHS     = 10        # head-only training; the pretrained bases stay frozen

TRAIN_SPLIT = 0.70
VAL_SPLIT   = 0.30

# The paper's dataset. On Kaggle it is already attached; anywhere else kagglehub
# downloads it once (~483 MB) and caches it.       pip install kagglehub
KAGGLE_INPUT = '/kaggle/input/behzad-safari-jalal/data'

if os.path.isdir(KAGGLE_INPUT):
    DATA_DIR = KAGGLE_INPUT
else:
    import kagglehub
    DATA_DIR = os.path.join(kagglehub.dataset_download('sinadunk23/behzad-safari-jalal'),
                            'data')

SPLIT_ROOT = '/kaggle/working/data_70_30' if os.path.isdir('/kaggle/working') else 'data_70_30'
TRAIN_DIR  = os.path.join(SPLIT_ROOT, 'train')
VAL_DIR    = os.path.join(SPLIT_ROOT, 'val')
MODEL_PATH = 'wheat_disease_cnn.keras'      # written by the main notebook


def class_sources(class_name):
    """Every folder under DATA_DIR holding images of this class."""
    wanted = class_name.lower().replace('_', ' ')
    found = sorted(os.path.join(parent, d)
                   for parent, dirs, _ in os.walk(DATA_DIR)
                   for d in dirs
                   if d.lower().replace('_', ' ') == wanted)
    assert found, f'No folder for class {class_name} under {DATA_DIR}'
    return found


CLASS_SOURCES = {name: class_sources(name) for name in CLASS_NAMES}

print('Data dir   :', DATA_DIR)
print('Split root :', SPLIT_ROOT)

## The 70:30 split

The same code and the same seed as the main notebook, so this reproduces the identical
partition. If `data_70_30/` already exists from that run, nothing is rewritten.

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.jfif', '.png', '.bmp', '.gif')


def list_images(directory):
    """Full paths of the image files directly inside `directory`."""
    if not os.path.isdir(directory):
        return []
    return [os.path.join(directory, f) for f in sorted(os.listdir(directory))
            if f.lower().endswith(IMG_EXTS)]


def file_md5(path):
    """Hash of the raw file bytes, used to spot byte-identical duplicates."""
    with open(path, 'rb') as fh:
        return hashlib.md5(fh.read()).hexdigest()


rng = np.random.RandomState(SEED)
print(f'Splitting {TRAIN_SPLIT:.0%}:{VAL_SPLIT:.0%} -> {SPLIT_ROOT}\n')

for name in CLASS_NAMES:
    pool = [path for folder in CLASS_SOURCES[name] for path in list_images(folder)]

    # Group byte-identical files so duplicates cannot straddle the split.
    groups = defaultdict(list)
    for path in pool:
        groups[file_md5(path)].append(path)
    keys = sorted(groups)
    rng.shuffle(keys)

    quota = int(round(TRAIN_SPLIT * len(pool)))
    assigned, side = 0, {}
    for key in keys:
        side[key] = 'train' if assigned < quota else 'val'
        assigned += len(groups[key])

    counts = {'train': 0, 'val': 0}
    for key in keys:
        target_dir = os.path.join(SPLIT_ROOT, side[key], name)
        os.makedirs(target_dir, exist_ok=True)
        for path in groups[key]:
            filename = os.path.basename(path)
            if filename.lower().endswith('.jfif'):
                filename = filename[:-len('.jfif')] + '.jpg'
            dst = os.path.join(target_dir, filename)
            if not os.path.exists(dst):
                try:
                    os.link(path, dst)          # hard link: costs no disk space
                except OSError:
                    shutil.copy2(path, dst)
            counts[side[key]] += 1
    print(f"{name:14s}train {counts['train']:5d}  val {counts['val']:5d}")

print(f'\nTrain dir: {TRAIN_DIR}')
print(f'Val dir  : {VAL_DIR}')

## The models

Every backbone is frozen, so only the new 3-class head learns. `make_generators`
rebuilds the data streams with that model's own preprocessing function.

In [ ]:
BASELINES = {
    'ResNet50':       (applications.ResNet50,       applications.resnet50.preprocess_input),
    'EfficientNetB0': (applications.EfficientNetB0, applications.efficientnet.preprocess_input),
    'MobileNetV2':    (applications.MobileNetV2,    applications.mobilenet_v2.preprocess_input),
}


def make_generators(preprocess):
    """The same 70:30 split, read with one model's own preprocessing.

    The custom CNN divides pixels by 255, but ResNet50 and MobileNetV2 expect
    different ranges, so each backbone gets its own generators.
    """
    gen = ImageDataGenerator(preprocessing_function=preprocess)
    train = gen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=CLASS_NAMES, shuffle=True, seed=SEED)
    val = gen.flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=CLASS_NAMES, shuffle=False)
    return train, val


def build_transfer_model(base_builder):
    """Frozen ImageNet backbone -> global average pooling -> 3-class softmax."""
    base = base_builder(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE)
    base.trainable = False                     # only the new head learns
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
    model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def trainable_params(model):
    return int(sum(np.prod(w.shape) for w in model.trainable_weights))


def milliseconds_per_image(model):
    """Average time to classify a single image, after one warm-up call."""
    one_image = np.zeros((1,) + INPUT_SHAPE, dtype='float32')
    model.predict(one_image, verbose=0)        # warm-up: not timed
    start = time.perf_counter()
    for _ in range(20):
        model.predict(one_image, verbose=0)
    return (time.perf_counter() - start) / 20 * 1000


print('Baselines:', list(BASELINES), f'| head-only training for {TL_EPOCHS} epochs')

## Train and measure

Roughly 10-20 minutes for all three on a Kaggle GPU, and hours on a CPU. The custom CNN
is measured first so every row of the table comes from the same validation set.

In [ ]:
comparison = []

# ---- the paper's CNN, trained by the main notebook ----
if os.path.exists(MODEL_PATH):
    cnn = tf.keras.models.load_model(MODEL_PATH)
    # The custom CNN was trained on pixels scaled to 0-1.
    _, cnn_val = make_generators(lambda x: x / 255.0)
    _, cnn_acc = cnn.evaluate(cnn_val, verbose=0)
    comparison.append({
        'name': 'Custom CNN (paper)',
        'val_acc': float(cnn_acc),
        'params': cnn.count_params(),
        'trainable': trainable_params(cnn),
        'ms': milliseconds_per_image(cnn),
    })
    print(f'Custom CNN (paper)  val_acc={cnn_acc:.4f}')
else:
    print(f'{MODEL_PATH} not found - run the main notebook first to include it.')

# ---- the pretrained backbones ----
for name, (base_builder, preprocess) in BASELINES.items():
    print(f'\n---------- {name} ----------')
    tl_train, tl_val = make_generators(preprocess)
    tl_model = build_transfer_model(base_builder)
    history = tl_model.fit(tl_train, validation_data=tl_val,
                           epochs=TL_EPOCHS, verbose=2)
    comparison.append({
        'name': name,
        'val_acc': float(max(history.history['val_accuracy'])),
        'params': tl_model.count_params(),
        'trainable': trainable_params(tl_model),
        'ms': milliseconds_per_image(tl_model),
    })
    print(f"{name}  best val_acc={comparison[-1]['val_acc']:.4f}")

print('\nAll baselines trained.')

## Results

In [ ]:
print(f"{'Model':22s}{'val acc':>9s}{'params':>13s}{'trainable':>12s}{'ms/image':>11s}")
print('-' * 67)
for row in comparison:
    print(f"{row['name']:22s}{row['val_acc']:9.4f}{row['params']:13,d}"
          f"{row['trainable']:12,d}{row['ms']:11.1f}")
print('-' * 67)

best     = max(comparison, key=lambda r: r['val_acc'])
smallest = min(comparison, key=lambda r: r['params'])
fastest  = min(comparison, key=lambda r: r['ms'])
print(f"Most accurate : {best['name']} ({best['val_acc']:.4f})")
print(f"Smallest      : {smallest['name']} ({smallest['params']:,} parameters)")
print(f"Fastest       : {fastest['name']} ({fastest['ms']:.1f} ms per image)")

names = [r['name'] for r in comparison]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(names, [r['val_acc'] for r in comparison], color='steelblue')
axes[0].set_xlim(0.0, 1.0)
axes[0].set_xlabel('Validation accuracy')
axes[0].set_title('Accuracy')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(names, [r['params'] / 1e6 for r in comparison], color='indianred')
axes[1].set_xlabel('Parameters (millions)')
axes[1].set_title('Model size')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## How to read this

- **Accuracy alone is not the answer.** If a frozen backbone matches the custom CNN with
  a fraction of the trainable parameters, that is an argument for transfer learning. If
  it does not, that is a genuine finding: this dataset is easy enough, and different
  enough from ImageNet, that features learned from scratch are competitive.
- **Trainable parameters** is the honest measure of what each model actually learned
  here. The frozen backbones only train a few thousand weights in the head.
- **Milliseconds per image** is measured on whatever hardware runs the notebook, at batch
  size 1. Compare the rows with each other, not against numbers from another machine.
- **Fine-tuning is the obvious next step.** Unfreezing the top few layers of the best
  backbone at a low learning rate usually adds accuracy. Say so in the write-up even if
  you do not run it - it shows you know the ceiling here is not the method's ceiling.
